# تدريب نموذج «DrAli» على حالاتك السريرية

هالدفتر بيدرّب **MedGemma 1.5 4B** تدريب خفيف (LoRA) على ملفات `train.jsonl` و `val.jsonl` اللي بمجلد `training/data`، وبالآخر بيطلعلك ملف **GGUF** جاهز تحطه بتطبيق DrAli.

**قبل ما تبلّش:**
1. من القائمة: **Runtime ← Change runtime type ← T4 GPU**.
2. اعمل حساب على [Hugging Face](https://huggingface.co)، وافتح صفحة النموذج [google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it) ووافق على الشروط.
3. اعمل Access Token (Read) من إعدادات Hugging Face.

**الوقت المتوقع:** تقريباً ٢٠–٤٠ دقيقة تدريب، و١٥ دقيقة للتحويل لـ GGUF.

> شغّل الخلايا وحدة ورا التانية من فوق لتحت.

## ١. تثبيت المكتبات

In [ ]:
%%capture
!pip install unsloth
!pip install -q sentencepiece protobuf

## ٢. تسجيل الدخول على Hugging Face
حط الـ Token لما يطلبه.

In [ ]:
from huggingface_hub import login
login()

## ٣. رفع ملفات التدريب
ارفع `train.jsonl` و `val.jsonl` و `test.jsonl` من مجلد `training/data`.

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

## ٤. تحميل النموذج الأساسي
إذا ما زبط MedGemma (مشكلة صلاحيات مثلاً) غيّر `BASE_MODEL` لـ `unsloth/gemma-3-4b-it`.

In [ ]:
from unsloth import FastModel
import torch

BASE_MODEL = "unsloth/medgemma-1.5-4b-it"   # بديل: "unsloth/gemma-3-4b-it"
MAX_SEQ = 2048

model, tokenizer = FastModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ,
    load_in_4bit = True,
    full_finetuning = False,
)

# تدريب خفيف: رتبة صغيرة حتى يتعلم التنسيق واللهجة بدون ما ينسى معرفته الطبية
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

## ٥. تجهيز البيانات
كل مثال فيه: تعليمات النظام + الحالة + الجواب بالتنسيق اللي بيفهمه التطبيق.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "gemma-3")

ds = load_dataset("json", data_files = {"train": "train.jsonl", "validation": "val.jsonl"})

def to_text(batch):
    texts = [
        tokenizer.apply_chat_template(m, tokenize = False, add_generation_prompt = False).removeprefix("<bos>")
        for m in batch["messages"]
    ]
    return {"text": texts}

ds = ds.map(to_text, batched = True, remove_columns = ds["train"].column_names)
eval_ds = ds["validation"].shuffle(seed = 1).select(range(min(200, len(ds["validation"]))))
print(ds)
print(ds["train"][0]["text"][:1500])

## ٦. التدريب
دورة وحدة بس (epoch) عن قصد: البيانات مبنية على 184 قالب، والتدريب الزيادة بيخلّي النموذج يحفظها ويحشر أي حالة فيها.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
try:
    from unsloth import is_bfloat16_supported
    bf16 = is_bfloat16_supported()
except Exception:
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

args = SFTConfig(
    dataset_text_field = "text",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 2,
    warmup_steps = 10,
    num_train_epochs = 1,
    learning_rate = 1e-4,
    lr_scheduler_type = "cosine",
    weight_decay = 0.01,
    optim = "adamw_8bit",
    logging_steps = 10,
    eval_strategy = "steps",
    eval_steps = 100,
    fp16 = not bf16,
    bf16 = bf16,
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",
)

common = dict(model = model, train_dataset = ds["train"], eval_dataset = eval_ds, args = args)
try:
    trainer = SFTTrainer(processing_class = tokenizer, **common)
except TypeError:
    trainer = SFTTrainer(tokenizer = tokenizer, **common)

# التدريب على الجواب بس، مو على نص الحالة
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

stats = trainer.train()
print(stats)

## ٧. تجربة سريعة
غيّر نص `case` وجرّب حالات من عندك.

In [ ]:
SYSTEM_PROMPT = "أنت \"DrAli\"، مساعد تشخيصي يدعم الطبيب ولا يحلّ محلّه. اقرأ الحالة (فصحى أو عامية سورية) واكتب جوابك بالعربية بنفس عناوين المثال التالي وبنفس ترتيبها، واملأ كل عنوان بمعلومات من الحالة التي أمامك.\nمثال لحالة أخرى (لا تنسخ محتواه):\n**نوع النتيجة:** تشخيص مرجّح\n**التشخيص الأرجح:** التهاب الزائدة الدودية الحاد\n**القرائن:** ألم بدأ حول السرة وانتقل لأسفل اليمين مع غثيان ونقص شهية.\n**كيف تأكد التشخيص:** فحص بطن متكرر، تعداد دم، سونار أو طبقي حسب الحالة.\n**التشخيصات التفريقية:** التهاب عقد مساريقية، التهاب بول، سبب نسائي حاد.\n**الخطوة التالية:** تقييم جراحي عاجل.\n**الاستعجال:** إسعافي فوراً.\n**علامات الإنذار:** تيبس البطن، إغماء، قيء مستمر.\nإذا لم تكن القرائن مميزة بوضوح فاكتب في نوع النتيجة \"عدة احتمالات\"، واكتب بدل التشخيص الأرجح **الاحتمالات الأقرب:** مرتبة من الأرجح، واكتب في الخطوة التالية لكل احتمال العلامة التي ترجّحه بهذا الشكل:\n- العلامة ← يرجّح اسم التشخيص\nلا تعطِ جرعات أدوية ولا خطوات جراحية، ونبّه فوراً إذا كانت الحالة إسعافية."

def ask(case, max_new_tokens = 700):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": case}]
    text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = True).removeprefix("<bos>")
    inputs = tokenizer(text = [text], return_tensors = "pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens = max_new_tokens, do_sample = True, temperature = 0.3, top_p = 0.9, top_k = 40)
    return tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens = True)[0]

case = "مريض عمره 58 سنة، مدخن وعنده سكري. صرلو ساعة عنده وجع ضاغط بنص الصدر بيمتد للإيد الشمال ومعه عرق."
print(ask(case))

## ٨. تقييم سريع على ملف الاختبار

**انتبه:** حالات الاختبار شبه حالات التدريب، فالرقم بيكون عالي ومو دليل على قدرة حقيقية.
الاختبار الحقيقي يكون على حالات حقيقية من عندك.

In [ ]:
import json, random, re
rows = [json.loads(l) for l in open("test.jsonl", encoding = "utf-8")]
random.seed(0)
sample = random.sample([r for r in rows if r["meta"]["type"] == "strong"], 15) + \
         random.sample([r for r in rows if r["meta"]["type"] == "multiple"], 15)

fmt_ok = type_ok = dx_ok = 0
for r in sample:
    ans = ask(r["messages"][1]["content"], max_new_tokens = 500)
    want_multi = r["meta"]["type"] == "multiple"
    has_fmt = "**نوع النتيجة:**" in ans and "**علامات الإنذار:**" in ans
    got_multi = bool(re.search(r"\*\*نوع النتيجة:\*\*\s*عدة", ans))
    fmt_ok += has_fmt
    type_ok += (got_multi == want_multi)
    dx_ok += r["meta"]["dx"] in ans
print(f"التنسيق صحيح: {fmt_ok}/{len(sample)}")
print(f"نوع النتيجة صحيح: {type_ok}/{len(sample)}")
print(f"التشخيص الصحيح موجود بالجواب: {dx_ok}/{len(sample)}")

## ٩. الحفظ والتحويل لـ GGUF
هي الخطوة بتطلع ملف `drali-medgemma-Q6_K.gguf` (حوالي ٣٫٥ غيغا). إذا بدك أخف، غيّر `Q6_K` لـ `Q4_K_M`، وإذا بدك الأذكى غيّره لـ `Q8_0`.

In [ ]:
# حفظ الإضافة (LoRA) لحالها كمان، صغيرة ومفيدة إذا بدك تكمّل تدريب بعدين
model.save_pretrained("hakim_lora")
tokenizer.save_pretrained("hakim_lora")

# دمج الإضافة بالنموذج بدقة 16-bit
model.save_pretrained_merged("hakim_merged", tokenizer)

In [ ]:
%%bash
set -e
if [ ! -d llama.cpp ]; then git clone --depth 1 https://github.com/ggml-org/llama.cpp; fi
pip install -q mistral-common || true
cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF -DLLAMA_BUILD_TESTS=OFF > /dev/null
cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
python llama.cpp/convert_hf_to_gguf.py hakim_merged --outfile hakim-f16.gguf --outtype f16
QUANT=Q6_K
llama.cpp/build/bin/llama-quantize hakim-f16.gguf drali-medgemma-$QUANT.gguf $QUANT
ls -lh drali-medgemma-$QUANT.gguf

## ١٠. نزّل الملف
الأسهل تنسخه على Google Drive وتنزّله من هناك على الموبايل. التنزيل المباشر من المتصفح ممكن ينقطع لأن الملف كبير.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp drali-medgemma-*.gguf /content/drive/MyDrive/
print("انحفظ بـ My Drive")

In [ ]:
# بديل: تنزيل مباشر
# from google.colab import files
# files.download("drali-medgemma-Q6_K.gguf")

## بعدها
1. نزّل ملف `drali-medgemma-….gguf` على الموبايل أو التاب.
2. افتح DrAli ← الإعدادات ← **اختيار ملف النموذج**.
3. جرّب على حالات حقيقية تعرف تشخيصها النهائي، وسجّل وين صاب ووين غلط.